# 00 - Configuración Inicial de Unity Catalog, Volumes y Gobierno (ElectroCasa)
Ejecutar este notebook antes de desplegar el bundle en cada entorno (`electrocasa_dev` / `electrocasa_prod`).

## Paso 1: Preparar `dev` 

In [0]:
dbutils.widgets.text("catalogo", "electrocasa_dev", "Nombre del Catálogo")
catalogo = dbutils.widgets.get("catalogo")

# 1. Crear catálogo independiente (detectando Storage Root del metastore si existe)
storage_root = None
for row in spark.sql("SHOW CATALOGS").collect():
    cat_actual = row[0]
    if cat_actual not in ("system", "samples", "__databricks_internal"):
        desc = (
            spark.sql(f"DESCRIBE CATALOG EXTENDED `{cat_actual}`")
            .filter("info_name = 'Storage Root'")
            .collect()
        )
        if desc and desc[0]["info_value"]:
            storage_root = desc[0]["info_value"].rsplit("/", 1)[0] + "/" + catalogo
            break

if storage_root:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalogo}` MANAGED LOCATION '{storage_root}'")
else:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalogo}`")

# 2. Crear esquemas Medallion y Volume de landing
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalogo}`.default")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalogo}`.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalogo}`.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalogo}`.gold")

spark.sql(f"CREATE VOLUME IF NOT EXISTS `{catalogo}`.default.vol_landing")

# 3. Crear las 6 subcarpetas de aterrizaje por fuente
fuentes = ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales"]
for fuente in fuentes:
    dbutils.fs.mkdirs(f"/Volumes/{catalogo}/default/vol_landing/{fuente}")

print(f"Catálogo '{catalogo}', esquemas y volúmenes configurados correctamente.")

## Asignación de Permisos RBAC (GRANT)

In [0]:
%sql
-- Permisos para Ingeniería
GRANT USE CATALOG ON CATALOG ${catalogo} TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${catalogo}.default TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${catalogo}.bronze TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${catalogo}.silver TO `electrocasa_ingenieria`;
GRANT ALL PRIVILEGES ON SCHEMA ${catalogo}.gold TO `electrocasa_ingenieria`;

-- Permisos para Analistas
GRANT USE CATALOG ON CATALOG ${catalogo} TO `electrocasa_analistas`;
GRANT USE SCHEMA ON SCHEMA ${catalogo}.gold TO `electrocasa_analistas`;
GRANT SELECT ON SCHEMA ${catalogo}.gold TO `electrocasa_analistas`;

-- Permisos para Auditoría
GRANT USE CATALOG ON CATALOG ${catalogo} TO `electrocasa_auditoria`;
GRANT USE SCHEMA ON SCHEMA ${catalogo}.gold TO `electrocasa_auditoria`;
GRANT SELECT ON SCHEMA ${catalogo}.gold TO `electrocasa_auditoria`;

In [0]:
%sql
-- Permisos de sistema para Auditoría 
-- GRANT USE CATALOG ON CATALOG system TO `electrocasa_auditoria`;
-- GRANT USE SCHEMA ON SCHEMA system.access TO `electrocasa_auditoria`;
-- GRANT SELECT ON TABLE system.access.audit TO `electrocasa_auditoria`;
-- GRANT SELECT ON TABLE system.access.table_lineage TO `electrocasa_auditoria`;
-- GRANT SELECT ON TABLE system.access.column_lineage TO `electrocasa_auditoria`;

In [0]:
%sql
-- Demostración de GRANT y REVOKE
GRANT SELECT ON SCHEMA ${catalogo}.silver TO `electrocasa_analistas`;
SHOW GRANTS ON SCHEMA ${catalogo}.silver;

REVOKE SELECT ON SCHEMA ${catalogo}.silver FROM `electrocasa_analistas`;
SHOW GRANTS ON SCHEMA ${catalogo}.silver;

In [0]:
%sql
-- Función de máscara dinámica para dni y salario
CREATE OR REPLACE FUNCTION ${catalogo}.silver.mask_dato_sensible(valor STRING)
RETURNS STRING
RETURN CASE 
    WHEN is_account_group_member('electrocasa_ingenieria') THEN valor
    ELSE '***'
END;

## Eliminación de recursos del Bundle y Eliminación de Catálogos y Volúmenes

In [0]:
%sh
#databricks bundle destroy -t dev --auto-approve
#databricks bundle destroy -t prod



In [0]:
# Limpieza completa de los catálogos del proyecto (borra schemas, tablas y volumes)
#spark.sql("DROP CATALOG IF EXISTS electrocasa_dev CASCADE")
#spark.sql("DROP CATALOG IF EXISTS electrocasa_prod CASCADE")

print("Pipelines y catálogos de dev y prod eliminados correctamente.")